## GMM (Manual Implementation)

In [1]:
import numpy as np

def gaussian_pdf(X, mean, cov):
    d = X.shape[1]
    diff = X - mean
    cov_inv = np.linalg.inv(cov + np.eye(d) * 1e-6)  # small regularization for stability
    cov_det = np.linalg.det(cov + np.eye(d) * 1e-6)
    coeff = 1 / (np.sqrt((2 * np.pi) ** d * cov_det))
    exponent = np.exp(-0.5 * np.sum(diff @ cov_inv * diff, axis=1))
    return coeff * exponent

def gmm_em(X, k, max_iters=100):
    n, d = X.shape
    np.random.seed(42)

    # initialize parameters
    pi = np.ones(k) / k                          # equal mixing weights
    means = X[np.random.choice(n, k, replace=False)]  # random initial means
    covs = [np.eye(d)] * k                       # identity covariance for each Gaussian

    for _ in range(max_iters):
        # E step — compute responsibilities
        r = np.zeros((n, k))
        for j in range(k):
            r[:, j] = pi[j] * gaussian_pdf(X, means[j], covs[j])
        r /= r.sum(axis=1, keepdims=True) + 1e-8  # normalize

        # M step — update parameters
        r_sum = r.sum(axis=0)
        pi = r_sum / n
        means = (r.T @ X) / r_sum[:, None]
        covs = [
            (r[:, j][:, None] * (X - means[j])).T @ (X - means[j]) / r_sum[j]
            for j in range(k)
        ]

    return r, means, pi

# Example usage
X_demo = np.array([[1.0,1.0],[1.2,1.1],[1.1,0.9],[8.0,8.0],[8.2,7.9],[7.9,8.1]])
r, means, weights = gmm_em(X_demo, k=2)

print("Cluster means:\n", means)
print("Mixing weights:", weights)
print("Responsibilities (probabilities per point):\n", r.round(3))

Cluster means:
 [[1.05 0.95]
 [1.2  1.1 ]]
Mixing weights: [0.33333333 0.16666667]
Responsibilities (probabilities per point):
 [[1. 0.]
 [0. 1.]
 [1. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
